---
## **0. 준비**
---

In [17]:
%%html
<style>
    body {
        /* change --vscode-font-family if it's different on your system */
        --vscode-font-family: "SeoulNamsanC";
    }
</style>

In [1]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import GoogleGenerativeAI

In [2]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [3]:
import config
config.set_environment()

---
## **LangChain의 빌딩 블록 살펴보기**
---

### **모델 인터페이스**

<hr style="border:0;border-top:1.5px dashed gray;">

#### **LLM 상호작용 패턴**
- 최근 LangChain 사용 사례를 보면 채팅 모델(ChatModel) 인터페이스를 선호하는 추세 -> 기존 LLM 인터페이스보다 채팅 모델 사용을 권장
- 아래 기본 원리를 이해하면 왜 채팅 모델로의 전환이 필요한지 명확히 알 수 있음

In [ ]:
# 방법 1

# OpenAI 모델 초기화
openai_llm = ChatOpenAI()                                                   #대화형 모델을 호출

# Gemini 모델 초기화
## gemini_pro = GoogleGenerativeAI(model="gemini-1.5-pro")

# 하나 또는 두 모델 모두 같은 인터페이스 사용 가능
response = openai_llm.invoke("전구에 대한 농담 말해줘.")                    #모든 공급자가 동일한 메서드를 사용하는 점이 중요

print(response)

In [ ]:
# 방법 2

question = "전구에 대한 농담을 한국어로 하나만 말해줘."

models = [
    "gpt-6-luna",
    "gpt-6.1-sol",
    "gpt-6-astra",
    "gpt-3.5-turbo",    
]

for model_name in models:
    print(f"\n===== {model_name} =====")
    llm = ChatOpenAI(model=model_name, use_responses_api=True,)             #use_responses_api=True 옵션에 대해서는 Craft 문서 참고 

    response = llm.invoke(question)
    #print(response)                                                        #응답 객체 전체: 콘텐츠, 모델 정보, 토큰 사용량 등, 용도는 응답 구조와 부가정보 확인
    #print(response.content)                                                #컨텐츠 원본: 문자열 또는 컨텐츠 블록 목록, 용도는 답변/추론/도구 관련 항목 확인
    print(response.text)                                                    #컨텐츠에서 추출한 답변 텍스트

<hr style="border:0;border-top:1.5px dashed gray;">

#### **개발 테스트**
- 개발 단계에서는 실제 API 호출 없이도 애플리케이션을 테스트 할 수 있음
- LangChain은 이러한 목적으로 `FakeListLLM`을 제공

In [ ]:
from langchain_community.llms import FakeListLLM

# 항상 동일한 응답을 반환하는 가짜 LLM 생성
fake_llm = FakeListLLM(responses=["이것은 테스트 응답입니다."])

result = fake_llm.invoke("어떤 인풋도 위 응답으로 도출")

print(result)

<hr style="border:0;border-top:1.5px dashed gray;">

#### **채팅 모델과 작업**
- 요즘 모델 공급자가 제공하는 API는 각 대화 턴을 별도의 잘 정리된 페이로드(payload) 형태로 처리하도록 되어 있음.
- 모델 공급자들은 일반적으로 채팅 기록을 서버 측에 저장하지 않으며, 매번 클라이언트로부터 전체 기록을 전송받은 후 서버에서 최종 프롬프트만 구조화하여 처리.
- LangChain은 채팅 모델에서도 이와 같은 패턴을 따르며, 역할과 내용이 명시된 구조화된 대화를 처리. 각 메시지는 다음 요소를 포함
    - 역할(누가 말하는지): 메세지 클래스로 정의됨(모든 메세지는 `BaseMessage`를 상속)
    - 내용(무엇을 말하는지)
- **메세지 유형**
    - **SystemMessage** : 모델의 기본 동작 방식 및 역할을 설정.
        - `SystemMessage(content= "You're a helpful programming assistant")`
    - **HumanMessage** : 질문, 명령어, 데이터 등의 사용자 입력.
        - `HumanMessage(content= "Factorial을 계산하는 함수를 작성해줘")`
    - **AIMessage** : 모델의 응답을 포함


In [ ]:
chat = ChatOpenAI(model = "gpt-6-luna", use_responses_api=True,)
messages = [
    SystemMessage(content="You're a helpful programming assistant"),
    HumanMessage(content="Factorial을 계산하는 파이썬 함수를 작성해줘. 설명 및 사용 예시도 함께 제공해야돼"),
]
response = chat.invoke(messages)
print(response.text)

<hr style="border:0;border-top:1.5px dashed gray;">

#### **추론 모델**

2026년 10월 13일 기준 변화는 다음과 같습니다.

| 구분 | 교재의 방식 | 최신 방식 |
|---|---|---|
| Claude | `budget_tokens`로 사고 토큰 예산 지정 | **최신 지원 모델은 적응형 사고와 `effort` 사용** |
| GPT | o3의 `reasoning_effort` | **최신 GPT에서도 추론 강도를 설정하며, 지원 단계는 모델별로 다름** |

Claude는 4.6에서 수동 예산 방식이 deprecated되었고, 4.7 이후 모델에서는 지원하지 않습니다. 따라서 “Claude는 토큰 예산, GPT는 강도”라는 교재의 구분이 최신 모델에는 그대로 적용되지 않습니다. [Claude 공식 문서](https://platform.claude.com/docs/en/build-with-claude/extended-thinking)

GPT 실습에는 `gpt-6-luna`를 사용했습니다. 이 모델은 `none`, `low`, `medium`, `high`, `xhigh`, `max`를 지원합니다. **강도는 정확한 토큰 수를 지정하는 값이 아닙니다.** [OpenAI 공식 문서](https://developers.openai.com/api/docs/models/gpt-6-luna)

추가한 셀의 핵심 설정은 다음과 같습니다.

```python
reasoning={"effort": effort, "summary": "auto"}
```

`reasoning_effort`와 같은 강도 설정을 딕셔너리에 넣고, 추론 요약도 함께 요청했습니다. **요약은 내부 사고 과정의 원문이 아니며, 응답에 없을 수도 있습니다.** [추론 안내](https://developers.openai.com/api/docs/guides/reasoning)

셀을 실행하면 동일한 다리 건너기 문제를 `low`와 `high`로 풀고 다음을 비교합니다.

- 최종 답변과 추론 요약
- 추론 토큰·전체 출력 토큰 사용량
- 응답 시간과 검산 기준인 17분

코드 구문과 실제 요청 설정 변환을 검증했습니다. **API 호출은 실행하지 않았으며**, 사용자가 셀을 실행하면 총 2회 호출되어 비용이 발생합니다.

In [ ]:
# GPT 추론 강도 비교 실습 (2026-10-13 공식 문서 기준)
# 실행하면 같은 질문으로 API를 2번 호출합니다. 추론 토큰도 출력 요금에 포함됩니다.
# Claude 최신 지원 모델: thinking={"type": "adaptive"}, output_config={"effort": "high"}
# GPT: reasoning_effort로 강도를 지정. 토큰 수를 지정하는 budget_tokens와는 다릅니다.
# GPT-6 Luna 지원 강도: none, low, medium(기본), high, xhigh, max

from time import perf_counter                                               #경과 시간을 정밀하게 측정하는 함수

# 기본 설정
MODEL = "gpt-6-luna"                                                        #비교할 모델 ID를 지정. 강도 비교에서는 모델을 동일하게 유지
EFFORTS = ["low", "high"]                                                   #반복문에서 low, high 순서로 실행할 추론 강도를 목록에 저장.
MAX_OUTPUT_TOKENS = 25_000                                                  #내부 추론 + 최종 답변의 합계 상한 (사고 예산이 아님)

# 여러 줄 문자열로 동일한 질문을 저장합니다. 따옴표 안의 각 줄은 모두 모델에게 보낼 문제 조건입니다.
question = """네 사람이 밤에 다리를 건너야 한다. 각자 다리를 건너는 데 1분, 2분, 5분, 10분이 걸린다.
다리에는 최대 두 사람만 동시에 올라갈 수 있다. 두 사람이 함께 건너면 느린 사람의 시간이 걸린다.
손전등은 하나이며, 건널 때마다 반드시 가지고 있어야 한다. 모두 처음에는 왼쪽에 있고, 손전등을 가지고 돌아오는 시간도 포함한다.
모두 오른쪽으로 이동하는 최소 총 시간과 이동 순서를 구하라. 한국어로 최종 답과 검증 가능한 간결한 근거를 제시하라."""

reasoning_results = {}                                                      #빈 딕셔너리 생성, 실행 후 reasoning_results["low"] 처럼 응답 재확인 가능, 재호출 없이 response/content/text를 살펴볼 수 있음

# 가동
for effort in EFFORTS:
    print(f"\n{'=' * 20} {MODEL} / effort={effort} {'=' * 20}")
    
    # 이번 반복에서 사용할 모델 객체를 만듭니다. 아직 API를 호출하지는 않습니다.
    llm = ChatOpenAI(
        model=MODEL,                                                        #위에서 지정한 모델 ID를 전달
        use_responses_api=True,                                             #추론 요약을 지원하는 Responses API로 요청하도록 지정
                
        reasoning={"effort": effort, "summary": "auto"},                    #원본 사고가 아닌 요약 요청, reasoning_effort=effort와 같은 강도 설정을 reasoning 딕셔너리에 지정, 이 환경에서는 두 설정을 함께 쓰면 딕셔너리가 우선하므로 한곳에 모음
                                                                            #effort로 추론 강도를, summary="auto"로 제공 가능한 추론 요약을 요청. 정확한 추론 토큰 수는 지정하지 않음.
        max_tokens=MAX_OUTPUT_TOKENS,                                       #Responses API의 max_output_tokens로 변환, 내부 추론과 최종 답변을 합친 출력 상한. 너무 작으면 최종 답변 전에 토큰이 소진될 수 있음.
        timeout=180,                                                        #요청 대기 제한을 180초로 지정
        max_retries=0,                                                      #요청 실패 시 LangChain/OpenAI 클라이언트의 자동 재시도를 하지 않음
    )

    # 추론-응답 시간 측정
    started = perf_counter()                                                #API 요청 직전의 시각을 기록, 달력 시각이 아니라 경과 시간 측정용 값.
    
    response = llm.invoke(question)                                         #실제 API를 호출하고 응답 객체를 저장, 이 줄에서 사용량에 따른 비용이 발생.
    
    elapsed = perf_counter() - started                                      #응답을 받은 시각에서 시작 시각을 빼 전체 대기 시간을 초 단위로 계산.
    
    # 응답 저장 및 출력
    reasoning_results[effort] = response                                    #현재 강도를 키로 응답을 보관, 다음 반복이 response를 덮어써도 이전 응답이 남음.

    print("\n[최종 답변]")
    print(response.text or "답변 텍스트 없음: 토큰 상한과 응답 상태를 확인하세요.") #답변 텍스트를 출력, 빈 문자열이면 or 오른쪽의 안내 문구를 출력.

    # 추론 요약 출력 구간 시작, 실제 내부 사고 원문은 공개되지 않음.
    print("\n[추론 요약 — 내부 사고 원문이 아닙니다]")                     
        # 조건에 맞는 요약만 모으는 리스트 컴프리헨션을 시작합니다.
        # [결과로 넣을 값 for 변수 in 반복할 대상 if 조건]
    summaries = [
        block["reasoning"]                                                  #조건을 통과한 콘텐츠 블록의 reasoning 문자열을 결과 목록에 넣음
        for block in response.content_blocks                                #응답을 표준화한 콘텐츠 블록 목록에서 블록을 하나씩 읽음
        if block.get("type") == "reasoning" and block.get("reasoning")]     #type이 reasoning이고 요약 내용이 비어 있지 않은 블록만 선택, get은 키가 없어도 오류 대신 None을 반환.
            
    print("\n".join(summaries) if summaries else                            #요약들을 줄바꿈으로 연결,
          "이 응답에는 공개된 추론 요약이 없습니다.")                       #빈 목록이면 요약이 없다는 안내를 출력.

    # 토큰 사용량 출력 구간
    usage = response.usage_metadata or {}                                   #토큰 사용량 딕셔너리를 가져옴, 정보가 None이면 빈 딕셔너리로 대체.
       
    reasoning_tokens = usage.get("output_token_details",                    #출력 토큰의 세부 정보에서 추론 토큰 수를 읽음, 세부 정보가 없으면 빈 딕셔너리를 사용.
                                 {}).get("reasoning")
        
    print(f"\n[사용량] 입력={usage.get('input_tokens', '미제공')}, "        #입력 토큰 수를 문자열에 넣음, 해당 키가 없으면 미제공으로 표시.
          f"출력(추론 포함)={usage.get('output_tokens', '미제공')}, "       #추론 토큰도 포함하는 출력 토큰 수를 앞 문자열에 이어 붙입니다.
          f"추론={reasoning_tokens if reasoning_tokens is not None else '미제공'}") #추론 토큰 수가 None이면 미제공을 표시, 0은 유효한 사용량이므로 그대로 표시합니다.
    
    # 응답 시간 및 상태 출력 구간
    print(f"[응답 시간] {elapsed:.2f}초")                                   #측정한 초를 소수점 둘째 자리까지 표시, 네트워크 대기 시간도 포함.
    print("[응답 상태]", response.response_metadata.get("status", "미제공"))#응답 메타데이터의 상태를 표시, status 항목이 없으면 미제공을 표시.

# 아래는 검산 기준과 주의 사항 안내
print("\n[검산 기준] 최소 17분: 1·2 건너기(2) → 1 돌아오기(1) → 5·10 건너기(10) → 2 돌아오기(2) → 1·2 건너기(2).")
print("한 번의 결과만으로 성능을 판단하지 마세요. 높은 강도가 항상 더 느리거나 정확한 것은 아닙니다.")


In [6]:
# 교재 방식
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are a problem-solving machine."),
    ("human", "{question}")
])

# reasoning_effort 매개변수를 설정해 초기화
chat = ChatOpenAI(
    model=MODEL,
    use_responses_api=True,
    reasoning={"effort": "high", "summary": "auto"},
    # or reasoning_effort="high", summary="auto"                            #reasoning 딕셔너리 대신 reasoning_effort와 summary를 직접 지정 가능
    max_tokens=MAX_OUTPUT_TOKENS,
    timeout=180,
    max_retries=0,
)

chain = template | chat
response = chain.invoke({"question": question})
print(response.text or "답변 텍스트 없음: 토큰 상한과 응답 상태를 확인하세요.")

**최소 총 시간은 17분**입니다.

| 이동 | 시간 |
|---|---:|
| 1분·2분 사람 → | 2분 |
| 1분 사람 ← | 1분 |
| 5분·10분 사람 → | 10분 |
| 2분 사람 ← | 2분 |
| 1분·2분 사람 → | 2분 |
| **합계** | **17분** |

**최적성 근거:** 느린 두 사람(5분·10분)을 함께 보내면, 빠른 두 사람이 손전등을 효율적으로 옮기는 데 드는 시간까지 합쳐 최소가 \(2+1+10+2+2=17\)분입니다. 두 사람을 따로 보내는 방식은 최소한 \(10+1+5+1+2=19\)분이 듭니다. 따라서 17분이 최소입니다.


#############################################################################

#### **스터디 Q&A: 프롬프트 템플릿, 체인, 메시지의 관계**
#############################################################################

**질문 1. `PromptTemplate`과 `ChatPromptTemplate`은 클래스인가? 두 클래스는 무엇이 다른가?**

둘 다 클래스입니다. **`from_template()`과 `from_messages()`는 템플릿 객체를 만드는 클래스 메서드**이고, `invoke()`는 객체에 입력을 전달해 실행하는 메서드입니다.

| 구분 | `PromptTemplate` | `ChatPromptTemplate` |
|---|---|---|
| 구성 | 하나의 문자열 템플릿 | 역할이 있는 메시지 템플릿들의 목록 |
| 변수 예시 | `"{topic}을 설명해줘"` | `("human", "{topic}을 설명해줘")` |
| `invoke()` 결과 | `StringPromptValue` | `ChatPromptValue` |
| 주된 특징 | 문자열의 빈칸을 채움 | 시스템·사용자 등의 역할을 유지하면서 빈칸을 채움 |

- `PromptTemplate.invoke()`가 곧바로 일반 문자열을 반환하는 것은 아닙니다. 반환한 프롬프트 값에서 `.to_string()`을 호출하면 문자열을 얻습니다. 
- `ChatPromptTemplate.invoke()`의 결과에서 `.to_messages()`를 호출하면 메시지 목록을 얻습니다.
- `PromptTemplate`도 `ChatOpenAI`와 연결할 수 있습니다. 다만 문자열로 만든 입력은 기본적으로 하나의 사용자 메시지로 전달되므로, 문자열에 `System:`이라고 적는 것만으로 실제 시스템 역할이 생기지는 않습니다. 역할 구분이 필요하면 `ChatPromptTemplate`을 사용합니다.

<hr style="border:0;border-top:1.5px dashed gray;">

```python
# 아래 예시는 템플릿만 채우므로 API 호출이나 모델 응답 생성은 하지 않음
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

text_template = PromptTemplate.from_template("{topic}을 쉽게 설명해줘.")
print(text_template.invoke({"topic": "팩토리얼"}).to_string())

chat_template = ChatPromptTemplate.from_messages([
    ("system", "너는 친절한 파이썬 선생님이다."),
    ("human", "{topic}을 쉽게 설명해줘."), ])
print(chat_template.invoke({"topic": "팩토리얼"}).to_messages())
```
<hr style="border:0;border-top:1.5px dashed gray;">

- 출력 결과는:
    - 팩토리얼을 쉽게 설명해줘.
    - [SystemMessage(content='너는 친절한 파이썬 선생님이다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='팩토리얼을 쉽게 설명해줘.', additional_kwargs={}, response_metadata={})]

#############################################################################

**질문 2. 직접 호출 실습도 템플릿과 모델을 연결한 체인으로 바꿀 수 있는가? 교재의 체인 예시와 같은 결과가 나오는가?**

가능합니다. `llm`과 `chat`은 변수 이름일 뿐이며, 두 변수에 같은 종류의 모델 객체를 넣었다면 연결 방식도 같습니다.

<hr style="border:0;border-top:1.5px dashed gray;">

```python
# question과 llm은 앞에서 만든 질문 문자열과 모델 객체
question_template = PromptTemplate.from_template("{question}")
chain = question_template | llm

# 아래 두 호출은 같은 질문을 전달하는 두 가지 방법입니다.
# 각각 실행하면 별도의 API 호출이므로, 비교하지 않을 때는 하나만 실행하세요.
direct_response = llm.invoke(question)
chain_response = chain.invoke({"question": question})
```
<hr style="border:0;border-top:1.5px dashed gray;">

- 체인의 `|`는 이 문맥에서 앞 단계의 결과를 다음 단계에 전달한다는 뜻입니다.
- 입력 딕셔너리 → 템플릿의 변수 채우기 → 완성된 프롬프트 → 모델 호출 → 응답
- 실제로는 다음과 같이 두 단계를 따로 실행하는 것과 같습니다.

<hr style="border:0;border-top:1.5px dashed gray;">

```python
prompt_value = question_template.invoke({"question": question})
chain_response = llm.invoke(prompt_value)
```
<hr style="border:0;border-top:1.5px dashed gray;">

- 같은 모델·설정·메시지 역할·문장을 전달하면 요청 내용은 같게 만들 수 있습니다. 
- 하지만 두 번 호출한 답변이 글자까지 같다고 보장되지는 않습니다. 템플릿에 새로운 시스템 지시나 문장을 넣으면 입력 자체도 달라집니다.
- 앞의 추론 실습에도 반복문 안에서 `chain = question_template | llm`을 만들고, `response = llm.invoke(question)` 대신 `response = chain.invoke({"question": question})`를 사용할 수 있습니다. 
- `reasoning` 등 추론 설정은 모델 객체 `llm`에 그대로 남고, 템플릿은 모델에 보낼 질문만 구성합니다.

#############################################################################

**질문 3. 앞에서 배운 `SystemMessage`·`HumanMessage`와 `ChatPromptTemplate`은 어떤 관계인가?**

- 메시지는 **내용이 채워진 실제 입력**, 템플릿은 **그 메시지를 반복해서 만들기 위한 틀**입니다.
- 즉, 템플릿을 invoke()하면 해당 템플릿에 맞는 메세지가 출력되는 형태임.
- **PromptTemplate.invoke()** 는 문자열 기반의 StringPromptValue를 반환합니다. 이를 채팅 모델에 전달하면 사용자 메시지 하나로 변환됩니다. (문자열을 확인하기 위해서는 .to_string() 메서드 사용)
- **ChatPromptTemplate.invoke()** 는 템플릿에 지정한 역할과 내용을 담은 ChatPromptValue를 반환합니다. 시스템·사용자 메시지를 넣었다면 둘 다 들어갑니다. (메세지 목록을 확인하기 위해서는 .to_message() 메서드 사용)

<hr style="border:0;border-top:1.5px dashed gray;">

```python
from langchain_core.messages import SystemMessage, HumanMessage

# 방법 A: 완성된 메시지를 직접 만듭니다.
messages = [
    SystemMessage(content="너는 친절한 파이썬 선생님이다."),
    HumanMessage(content="팩토리얼을 쉽게 설명해줘."),
]

# 방법 B: 사용자 메시지에 바뀔 부분을 두고, 나중에 채웁니다.
template = ChatPromptTemplate.from_messages([
    ("system", "너는 친절한 파이썬 선생님이다."),
    ("human", "{topic}을 쉽게 설명해줘."),])
filled_messages = template.invoke({"topic": "팩토리얼"}).to_messages()

# messages와 filled_messages는 같은 역할과 내용을 갖습니다.
assert messages == filled_messages

# 모델 객체 chat이 준비되어 있다면 다음 두 방식 중 하나로 호출합니다.
# response = chat.invoke(messages)
# chain = template | chat
# response = chain.invoke({"topic": "팩토리얼"})
```
<hr style="border:0;border-top:1.5px dashed gray;">

- 현재 교재 방식 셀에서 `("system", "...")`은 시스템 메시지를 만드는 템플릿이고, `("human", "...")`은 사용자 메시지를 만드는 템플릿입니다. `{topic}`을 채운 뒤에는 각각 `SystemMessage`와 `HumanMessage`가 됩니다. 모델이 반환한 답변은 `AIMessage`입니다.
- 직접 호출 셀에도 시스템·사용자 메시지를 전달할 수 있습니다. 템플릿이 있어야만 메시지 역할을 사용할 수 있는 것은 아닙니다.

<hr style="border:0;border-top:1.5px dashed gray;">

```python
# 기존 반복문의 호출 부분을 아래처럼 바꾸면 교재 방식과 같은 메시지를 구성합니다.
response = llm.invoke([
    SystemMessage(content="You are a problem-solving machine."),
    HumanMessage(content=question),])
```
<hr style="border:0;border-top:1.5px dashed gray;">
위 방식과 교재의 체인 방식은 같은 메시지를 만들 수 있습니다. `llm`도 `high`로 설정하고 모델·출력 상한 등 조건을 맞춰야 비교가 가능합니다.

- 시스템 메시지는 역할·응답 방식 등의 지시를 전달하고, 사용자 메시지는 이번 질문을 전달합니다. 
- `ChatPromptTemplate`은 이 구조를 유지하면서 입력만 바꿀 수 있게 합니다. 

**공식 참고 자료**

- [LangChain: 역할별 ChatPromptTemplate과 체인 예시](https://docs.langchain.com/oss/python/integrations/chat/amazon_nova)
- [LangChain: PromptTemplate과 모델 연결 예시](https://docs.langchain.com/oss/python/integrations/llms/openai) — 템플릿 연결 방식 참고용이며, 이 페이지의 구형 모델을 실습에 사용할 필요는 없습니다.

#############################################################################

#### **모델 동작 제어**
- LLM은 생성 동작을 세밀하게 제어할 수 있는 여러 매개변수를 제공하며, 구체적인 구현은 공급자마다 다름

| 매개변수 | 설명 | 일반적인 범위 | 최상의 대상 |
|---|---|---|---|
| **Temperature** | 텍스트 생성 시 무작위성 제어 | 0.0 - 1.0 | 낮음(0.0 - 0.3) : 사실적 작업, Q&A<br>높음(0.7+) : 창의적 글쓰기, 브레인 스토밍 |
| **Top-k** | 가장 확률이 높은 k개의 토큰으로 선택 범위를 제한 | 1 - 100 | 낮은 값(1-10) : 집중된 출력<br>높은 값 : 다양한 완성도 |
| **Top-p** | 누적 확률이 임계값에 도달할 때까지 토큰 후보를 고려 | 0.0 - 1.0 | 낮은 값(0.5) : 더 집중된 출력<br>높은 값(0.9) : 더 다양한 아이디어와 창의적 응답 |
| **Max tokens** | 최대 응답 길이 제한 | 모델별 | 비용 관리 및 장황한 출력 방지 |
| **Presence/frequency penalties** | 이미 나타난 토큰에 패널티를 부여해 반복되는 것을 억제 | -2.0 - 2.0 | 반복이 바람직하지 않은 긴 글 생성시에 유용 |
| **Stop sequnences** | 모델에 생성 중지 시점을 알려줌 | 사용자 정의 문자열 | 정확한 생성 종료 지점 제어 |

- **Temperature + Top-k/Top-p** : 먼저 Top-k/Top-p가 토큰 분포를 필터링하고, 그 후 Temperature가 필터링된 집합 내에서 무작위성을 조절.
- **Penalties + Temperature** : 높은 Temperature와 낮은 Penalties를 함께 사용하면 창의적이지만 반복적인 텍스트가 생성될 수 있음.
- 엔터프라이즈 어플리케이션에서는 일관성과 정확성이 가장 중요하므로 이를 위해 낮은 Temperature(0.0-0.3)와 중간 범위의 Top_p(0.5-0.7)을 조합하는 것이 효과적임.

In [4]:
# 예시
from langchain_openai import OpenAI

# 사실 중심의 일관된 응답
fractual_llm = OpenAI(temperature=0.1, max_tokens=256)

# 창의적 브레인스토밍
creative_llm = OpenAI(temperature=0.8, top_p=0.95, max_tokens=512)

---
### **프롬프트와 템플릿**
**LangChain의 프롬프트 관리 시스템**은 크게 네 가지 기능을 갖추고 있음
- 동적 프롬프트 생성을 위한 **템플릿 시스템**
- 변경사항을 추적할 수 있는 **프롬프트 관리 및 버전 관리 기능**
- 모델 성능을 향상하기 위한 **퓨샷(few-shot) 예시 관리**
- 신뢰할 수 있는 결과를 도출하기 위한 **출력 파싱 및 검증**

> **LangChain의 프롬프트 템플릿은 정적 텍스트를 변수 치환이 가능한 동적 프롬프트로 변환**

> **또한, 템플릿의 주된 가치는 프롬프트를 LangChain 체인에서 공통 인터페이스로 다룰 수 있게 하는데 있음**

In [8]:
# 1. 동적 프롬프트 생성 함수를 직접 정의해서 활용
def generate_prompt(question, context=None):
    if context:
        return f"Context information: {context}\n\nAnswer this question concisely: {question}"
    return f"Answer this question concisely: {question}"

prompt_text = generate_prompt("프랑스의 수도는?", context="유럽의 국가 수도를 묻는 질문입니다.")
print(prompt_text)

# 모델에 직접 연결해야함
llm = ChatOpenAI(model='gpt-6-luna')
response = llm.invoke(prompt_text)
print(response.text)

Context information: 유럽의 국가 수도를 묻는 질문입니다.

Answer this question concisely: 프랑스의 수도는?
파리입니다.


In [9]:
# 2.LangChain의 PromptTemplate 활용
from langchain_core.prompts import PromptTemplate

# 한번 정의해 여러 곳에서 재사용
question_template = PromptTemplate(
    input_variables=["question", "context"],
    template="Context information: {context}\n\nAnswer this question concisely: {question}"
)

prompt_text = question_template.format(question="프랑스의 수도는?", context="유럽의 국가 수도를 묻는 질문입니다.")
print(prompt_text)

# 다른 langchain 구성요소와 연결 가능
chain = question_template | llm
response = chain.invoke({"question": "프랑스의 수도는?", "context": "유럽의 국가 수도를 묻는 질문입니다."})
print(response.text)

Context information: 유럽의 국가 수도를 묻는 질문입니다.

Answer this question concisely: 프랑스의 수도는?
프랑스의 수도는 파리입니다.


<hr style="border:0;border-top:1.5px dashed gray;">

#### **채팅 프롬프트 템플릿**
- 채팅 모델의 경우, 다양한 역할이 포함된 더욱 구조화된 프롬프트를 생성할 수 있음

In [7]:
template = ChatPromptTemplate.from_messages([
    ("system", "너는 한국어를 불어로 번역하는 번역기야."),
    ("human", "다음 말을 불어로 번역해줘: {text}")])

chat = ChatOpenAI(
    model="gpt-6-luna",
    use_responses_api=True,
    reasoning_effort="low", 
)

# 순차 적용 결과
formatted_messages = template.format_messages(text= "실존은 본질에 선행한다.")
response_1 = chat.invoke(formatted_messages)
print(response_1.text)

# chain으로 연결한 결과
chain = template | chat
response_2 = chain.invoke({"text": "실존은 본질에 선행한다."})
print(response_2.text)

L’existence précède l’essence.
L’existence précède l’essence.


---
### **LangChain 표현 언어(LECL)**
- LECL은 LangChain 기반 LLM 어플리케이션 개발 방식을 혁신적으로 진화시킨 기술
- 개발자는 원하는 최종 결과만 정의하면 실행 세부 사항은 LangChain이 자동으로 처리
- 파이프 연산자를 사용해 컴퍼넌트들을 연결하면 처리 흐름이 자동으로 구성 -> **Runnable 인터페에스 덕분**
- Runnable Interface :
    - 모든 컴퍼넌트가 표준화된 방식으로 입력을 처리하고 출력을 생성
    - 일관된 메서트 세트를 제공 :
        - `invoke()`: 단일 입력을 동기적으로 처리하고 출력을 반환
        - `stream()`: 출력이 생성되는 동안 실시간으로 스트림 형태로 전달
        - `batch()`: 다중 입력 병렬 처리
        - `ainvoke()`, `abatch()`, `astream()`: 위 메서드들의 비동기 버전

<hr style="border:0;border-top:1.5px dashed gray;">

#### **LECL을 이용한 간단한 워크플로**

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 컴포넌트 생성
prompt = PromptTemplate.from_template("다음에 대한 농담해줘 : {topic}")
llm = ChatOpenAI(model="gpt-6-luna", use_responses_api=True, reasoning_effort="low")
output_parser = StrOutputParser()

# LECL 체인 구성
chain = prompt | llm | output_parser

# 한 번의 호출로 워크플로 실행
result = chain.invoke({"topic": "프로그래밍"})
print(result)

프로그래머가 바다에 가면 가장 먼저 하는 일은?

파도를 **처리(handling)**하는 거야. 파도마다 예외가 생길 수 있으니까!


<hr style="border:0;border-top:1.5px dashed gray;">

#### **복잡한 체인의 예**

In [26]:
# 스토리 생성과 분석 사례 : 고급화된 패턴
"""
1. LLM 호출을 통해 초기 컨텐츠를 생성
2. 생성된 출력을 두번째 LLM 호출의 입력값으로 전달
3. 전체 체인 실행 과정에서 데이터의 지속성 유지 및 변환을 처리
"""

# 모델 초기화 
llm_1 = ChatOpenAI(
    model= "gpt-6-luna",
    reasoning_effort="none",                                                 #생략과 다름
    use_responses_api= True, temperature=0.8, top_p=0.95, max_tokens=512)
    # 추론 강도가 none이 아닐때는 temperature, top_p등을 제거해야함.

llm_2 = ChatOpenAI(
    model = "gpt-6-sol",
    reasoning_effort="medium",
    use_responses_api= True)
    
# 첫 번째 체인 : 이야기 생성
story_prompt = PromptTemplate.from_template("다음에 대한 스토리 작성 : {topic}")
story_chain = story_prompt | llm_1 | StrOutputParser()

# 두 번쩨 체인 : 이야기 분석
analysis_prompt = PromptTemplate.from_template(
    "이 이야기의 분위기를 분석해줘:\n{story}")
analysis_chain = analysis_prompt | llm_2 | StrOutputParser()

In [27]:
# 체인 결합
story_with_analysis = story_chain | analysis_chain

# 결합 체인 실행
story_analysis = story_with_analysis.invoke({"topic": "비 오는날"})

print("\nAnalysis:", story_analysis)


Analysis: 이 이야기는 **아쉬움에서 시작해 포근하고 희망적인 분위기로 바뀝니다.**

처음에는 소풍날 비가 내려 지우가 실망하고, 창문을 두드리는 빗소리도 그 마음을 더 쓸쓸하게 만듭니다. 하지만 민호가 찾아오면서 분위기가 달라집니다. 함께 우산을 쓰고 걷는 모습, 도서관의 따뜻한 코코아 냄새와 그림책 읽기 행사는 비 오는 날을 아늑하게 느끼게 합니다.

마지막의 “지우의 마음에는 작은 햇살이 비치고 있었다”는 표현은 **날씨는 그대로여도 지우의 마음은 밝아졌음**을 보여줍니다. 전체적으로 예상과 다른 하루에서도 즐거움을 발견하는, 따뜻하고 긍정적인 분위기의 이야기입니다.


> 위 방법은 작동하지만 한계가 있음 -> 분석 결과만 저장되고 **원본 스토리는 사라짐**

In [ ]:
# 1. RunnablePassthrough.assign으로 데이터 보존

from langchain_core.runnables import RunnablePassthrough
enhanced_chain = RunnablePassthrough.assign(
    story=story_chain                                                       #생성된 내용을 story 키로 추가
).assign(
    analysis=analysis_chain                                                 #스토리 분석 결과를 analysis 키로 추가
)
# 체인 실행
result = enhanced_chain.invoke({'topic':'비 오는날'})
print(result.keys())

dict_keys(['topic', 'story', 'analysis'])


In [ ]:
# 내용 확인
print(result['story'], "\n")
print("분석:\n", result['analysis'])

> 위 사례에서 딕셔너리를 직접 구성하여 출력 구조를 더 세밀하게 제어할 수 있음

In [ ]:
# 2. 딕셔너리 구성 방식 대안
from operator import itemgetter

manual_chain = (
    RunnablePassthrough() |                                                 #입력 그대로 전달
    {
        "story" : story_chain,                                              #스토리 결과 추가
        "topic" : itemgetter("topic")                                       #기존 topic 보존
    } |
    RunnablePassthrough().assign(analysis=analysis_chain)
)
result_manual = manual_chain.invoke({"topic":'비 오는날'})
print(result_manual.keys())

dict_keys(['story', 'topic', 'analysis'])


In [ ]:
# # 내용 확인
print(result_manual['story'], "\n")
print("분석:\n", result_manual['analysis'])

> 딕셔너리 변환을 사용해 LCEL 단축 표기법으로 위를 단순화할 수 있음

In [ ]:
# 3. 간단한 딕셔너리 구성
simple_dict_chain_corrected = story_chain | {
    "story": RunnablePassthrough(),                                      #스토리 출력을 'story'로 전달
    "analysis" : analysis_chain
}

# analysis_chain은 예상대로 {'story' :'...(생성된 이야기)'}을 수신
result_corrected = simple_dict_chain_corrected.invoke({'topic': '비 오는날'})
print(result_corrected.keys())

dict_keys(['story', 'analysis'])


In [ ]:
# 내용 확인
print(result_corrected['story'], "\n")
print("분석:\n", result_corrected['analysis'])

- **복수 LLM 호출** : 단일 프롬프트가 아닌, 여러 LLM 상호작용을 연결하여 파이프라인을 구성
- **데이터 변환** : `RunnablePassthrough`와 `itemgetter`를 활용하여 데이터를 효율적으로 관리하고 변환
- **딕셔너리 유지** : 단일 값을 넘어 전체 컨텍스트를 체인 전반에 걸쳐 유지
- **구조화된 출력** : 단순 문자열이 아닌 구조화된 딕셔너리 형태의 출력을 생성

> **LCEL**은 복잡한 워크플로를 우아하게 처리할 수 있지만, **상태 관리와 고급 분기 로직 같은 고급 기능은 한계가 있음 -> LangGraph 필요**